# Test AlexNet (Cochleogram input)

This notebook mirrors `03_test_baselinecnn.ipynb` but evaluates the **AlexNet** architecture
as described in the reference paper (Mang et al. 2024, *Sensors* 24, 682).

Architecture (matching Table 2 of the paper):
- 5 convolutional layers (11×11, 5×5, 3×3, 3×3, 3×3)
- 3 pooling layers (3×3, 2×2)
- ReLU activations
- 3 fully connected layers (4096 → 4096 → num_classes)

Training hyperparameters (matching Section 4.4 of the paper):
- Optimizer: ADAM
- Learning rate: 0.001
- Batch size: 16
- Epochs: 30
- 10-fold patient-wise cross-validation

In [1]:
import sys
import torch
sys.path.insert(0, '../src')

from cochleogram_vit.models.alexnet import AlexNet

# Setup parameters matching our cochleogram configuration
BATCH_SIZE = 4
IN_CHANNELS = 1
IMAGE_SIZE = 128
NUM_CLASSES = 4

print(f"Testing environment initialized!")
print(f"PyTorch version: {torch.__version__}")

Testing environment initialized!
PyTorch version: 2.11.0+cu128


## 1. Instantiate the Model
We verify that the model correctly instantiates and check the parameter count.

In [2]:
# Initialize the model
model = AlexNet(in_channels=IN_CHANNELS, num_classes=NUM_CLASSES)

def count_parameters(m):
    return sum(p.numel() for p in m.parameters() if p.requires_grad)

print(model)
print("=" * 60)
print(f"Total trainable parameters: {count_parameters(model):,}")

AlexNet(
  (features): Sequential(
    (0): Conv2d(1, 96, kernel_size=(11, 11), stride=(4, 4), padding=(2, 2))
    (1): ReLU(inplace=True)
    (2): MaxPool2d(kernel_size=3, stride=2, padding=0, dilation=1, ceil_mode=False)
    (3): Conv2d(96, 256, kernel_size=(5, 5), stride=(1, 1), padding=(2, 2))
    (4): ReLU(inplace=True)
    (5): MaxPool2d(kernel_size=3, stride=2, padding=0, dilation=1, ceil_mode=False)
    (6): Conv2d(256, 384, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (7): ReLU(inplace=True)
    (8): Conv2d(384, 384, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (9): ReLU(inplace=True)
    (10): Conv2d(384, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (11): ReLU(inplace=True)
    (12): MaxPool2d(kernel_size=3, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (avgpool): AdaptiveAvgPool2d(output_size=(3, 3))
  (classifier): Sequential(
    (0): Dropout(p=0.5, inplace=False)
    (1): Linear(in_features=2304, out_features=4096, bias=True)
 

## 2. Forward Pass Test (Dummy Batch)
We create a synthetic batch matching the expected output of `CochleogramTransform`,
and pass it through the model. If shapes line up, AlexNet is compatible with the
existing `trainer.py` engine.

In [3]:
# Create a dummy batch resembling a cochleogram tensor: (B, C, H, W)
dummy_input = torch.randn(BATCH_SIZE, IN_CHANNELS, IMAGE_SIZE, IMAGE_SIZE)
print(f"Input shape: {dummy_input.shape}")

model.eval()
with torch.no_grad():
    logits = model(dummy_input)

print(f"Output shape: {logits.shape}")

assert logits.shape == (BATCH_SIZE, NUM_CLASSES), (
    f"Output shape mismatch! Expected {(BATCH_SIZE, NUM_CLASSES)}, got {logits.shape}"
)
print("\nSuccess!")
print("AlexNet accepted the (B, 1, 128, 128) input and outputted (B, 4) logits.")

print("\nSample Logits (Raw scores):")
print(logits)

print("\nSample Probabilities (Softmaxed):")
print(torch.softmax(logits, dim=1))

Input shape: torch.Size([4, 1, 128, 128])
Output shape: torch.Size([4, 4])

Success!
AlexNet accepted the (B, 1, 128, 128) input and outputted (B, 4) logits.

Sample Logits (Raw scores):
tensor([[0.0063, 0.0196, 0.0188, 0.0079],
        [0.0078, 0.0187, 0.0181, 0.0067],
        [0.0086, 0.0189, 0.0185, 0.0083],
        [0.0062, 0.0195, 0.0182, 0.0081]])

Sample Probabilities (Softmaxed):
tensor([[0.2483, 0.2516, 0.2514, 0.2487],
        [0.2488, 0.2515, 0.2513, 0.2485],
        [0.2488, 0.2513, 0.2512, 0.2487],
        [0.2483, 0.2516, 0.2513, 0.2488]])


In [4]:
# ---------------------------------------------------------
# 3. 10-Fold CV — iter5 = iter1 setup (1-ch grayscale, Sco=50.14%) + resilience
# ---------------------------------------------------------
# Reverted from iter4 (3-ch viridis). Iter4 was eating 2.4 GB per fold and
# filling the disk; iter1's settings already match the paper to within 2.3 pts.
#
# Resilience changes vs iter1:
#   - save_every=999 -> only `best.pt` written (no epoch_NNN.pt every 5 epochs)
#   - per-fold checkpoint dir (../checkpoints/fold_NN/) so folds don't overwrite
#   - shutil.rmtree the fold dir after eval (zero residual disk between folds)
#   - per-fold predictions cached to ../runs/iter5_preds/fold_NN.npz (resumable)
# ---------------------------------------------------------

import os
import shutil
from pathlib import Path

from sklearn.model_selection import GroupKFold
from torch.utils.data import DataLoader, Subset
import torch
import pandas as pd
import numpy as np

from cochleogram_vit.utils.config import get_device, load_config, seed_everything
from cochleogram_vit.training.trainer import Trainer

cfg = load_config('../configs/default.yaml')
cfg["training"]["learning_rate"] = float(cfg["training"]["learning_rate"])
cfg["training"]["weight_decay"] = float(cfg["training"]["weight_decay"])
cfg["data"]["raw_dir"] = f"../{cfg['data']['raw_dir']}"
cfg["data"]["processed_dir"] = f"../{cfg['data']['processed_dir']}"
cfg["logging"]["log_dir"] = f"../{cfg['logging']['log_dir']}"
cfg["logging"]["save_dir"] = f"../{cfg['logging']['save_dir']}"

# Paper hyperparameters (Mang et al. 2024, Section 4.4)
cfg["training"]["epochs"] = 30
cfg["training"]["batch_size"] = 16
cfg["training"]["learning_rate"] = 1e-3
cfg["training"]["num_workers"] = 0

# Skip periodic checkpoint saves; only `best.pt` will be written
cfg["logging"]["save_every"] = 999

seed_everything(cfg["training"]["seed"])
device = get_device(cfg["training"]["device"])
print(f"Training on device: {device}")


class PrecomputedCochleogramDataset(torch.utils.data.Dataset):
    def __init__(self, metadata_csv: str):
        self.df = pd.read_csv(metadata_csv)
        self.df['npy_path'] = self.df['npy_path'].apply(lambda x: '../' + x)
        self.df['patient_id'] = self.df['npy_path'].apply(
            lambda x: os.path.basename(x).split('_')[0]
        )

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        cg = np.load(row['npy_path'])
        return torch.from_numpy(cg).unsqueeze(0), int(row['label'])


dataset = PrecomputedCochleogramDataset(f"{cfg['data']['processed_dir']}/metadata.csv")
print(f"Total cochleograms: {len(dataset)}")
print(f"Unique patients : {dataset.df['patient_id'].nunique()}")

groups = dataset.df['patient_id'].values
gkf = GroupKFold(n_splits=10)

PRED_DIR = Path('../runs/iter5_preds')
PRED_DIR.mkdir(parents=True, exist_ok=True)

global_targets, global_preds = [], []

for fold, (train_idx, val_idx) in enumerate(gkf.split(dataset, groups=groups)):
    fold_path = PRED_DIR / f'fold_{fold:02d}.npz'

    # Resume: load cached predictions if this fold already finished
    if fold_path.exists():
        d = np.load(fold_path)
        global_preds.extend(d['preds'].tolist())
        global_targets.extend(d['targets'].tolist())
        print(f"FOLD {fold+1}/10: loaded cached predictions ({len(d['preds'])} samples)")
        continue

    print(f"\n{'='*40}\nFOLD {fold+1}/10\n{'='*40}")
    print(f"Train: {len(train_idx)} | Val: {len(val_idx)}")

    train_loader = DataLoader(
        Subset(dataset, train_idx),
        batch_size=cfg["training"]["batch_size"], shuffle=True, pin_memory=True,
    )
    val_loader = DataLoader(
        Subset(dataset, val_idx),
        batch_size=cfg["training"]["batch_size"], shuffle=False, pin_memory=True,
    )

    fold_save_dir = f"../checkpoints/fold_{fold:02d}"
    cfg_fold = {**cfg}
    cfg_fold["logging"] = {**cfg["logging"], "save_dir": fold_save_dir}

    # 1-channel grayscale input (iter1 setup)
    model = AlexNet(in_channels=1, num_classes=cfg["model"]["num_classes"]).to(device)

    trainer = Trainer(
        model=model, train_loader=train_loader, val_loader=val_loader,
        cfg=cfg_fold, device=device,
    )
    trainer.fit()

    # Evaluate this fold's val split
    model.eval()
    fold_preds, fold_targets = [], []
    with torch.no_grad():
        for x, y in val_loader:
            x = x.to(device, non_blocking=True)
            logits = model(x)
            fold_preds.extend(logits.argmax(-1).cpu().numpy())
            fold_targets.extend(y.numpy())

    np.savez(fold_path, preds=np.asarray(fold_preds), targets=np.asarray(fold_targets))
    global_preds.extend(fold_preds)
    global_targets.extend(fold_targets)
    print(f"  saved -> {fold_path}")

    # Reclaim disk: drop this fold's checkpoint dir now that preds are safe
    shutil.rmtree(fold_save_dir, ignore_errors=True)
    print(f"  removed {fold_save_dir} (preds already cached)")

print(f"\n10-Fold CV done. Total predictions: {len(global_preds)}")
print(f"Per-fold predictions cached in: {PRED_DIR}")


Training on device: cuda
Total cochleograms: 6898
Unique patients : 126
FOLD 1/10: loaded cached predictions (691 samples)
FOLD 2/10: loaded cached predictions (691 samples)
FOLD 3/10: loaded cached predictions (691 samples)
FOLD 4/10: loaded cached predictions (691 samples)
FOLD 5/10: loaded cached predictions (690 samples)
FOLD 6/10: loaded cached predictions (690 samples)
FOLD 7/10: loaded cached predictions (689 samples)
FOLD 8/10: loaded cached predictions (689 samples)
FOLD 9/10: loaded cached predictions (685 samples)
FOLD 10/10: loaded cached predictions (691 samples)

10-Fold CV done. Total predictions: 6898
Per-fold predictions cached in: ../runs/iter5_preds


In [5]:
# ---------------------------------------------------------
# 4. Aggregated ICBHI Metric Evaluation
# ---------------------------------------------------------
# Using the exact methodology from the paper:
# Creating an "Aggregated Confusion Matrix" from the 10 folds
# and calculating the official 4-class ICBHI Score.
# ---------------------------------------------------------

import numpy as np
from sklearn.metrics import confusion_matrix

print("Calculating the Aggregated Confusion Matrix (All 10 Folds combined)...")
n_classes = 4

cm_total = confusion_matrix(global_targets, global_preds, labels=list(range(n_classes)))

print("\n--- AGGREGATED 4x4 CONFUSION MATRIX ---")
print("Target rows vs Predict columns (0: Normal, 1: Crackle, 2: Wheeze, 3: Both)")
print(cm_total)

# --- EXACT ICBHI PAPER LOGIC (4-Class Scoring) ---
TN = cm_total[0, 0]
TP = cm_total[1, 1] + cm_total[2, 2] + cm_total[3, 3]
FP = cm_total[0, 1:].sum()
FN = cm_total[1:, 0].sum()

print("\n--- 4-CLASS EVALUATION METRICS ---")
print(f"True Negatives (TN) : {TN} (Healthy recognized as Healthy)")
print(f"True Positives (TP) : {TP} (Anomalies correctly classified into their EXACT class)")
print(f"False Positives (FP): {FP} (Healthy wrongly flagged as Anomaly)")
print(f"False Negatives (FN): {FN} (Anomalies missed or misclassified)")

Sen = TP / (TP + FN + 1e-8)
Spe = TN / (TN + FP + 1e-8)
Score = (Sen + Spe) / 2.0

print("\n--- FINAL COMPUTED METRICS vs PAPER ---")
print(f"Computed Sensitivity (Sen): {Sen * 100:.2f}%  | Target vs AlexNet: 45.12%")
print(f"Computed Specificity (Spe): {Spe * 100:.2f}%  | Target vs AlexNet: 59.74%")
print(f"Computed Score (Sco)      : {Score * 100:.2f}%  | Target vs AlexNet: 52.43%")

Calculating the Aggregated Confusion Matrix (All 10 Folds combined)...

--- AGGREGATED 4x4 CONFUSION MATRIX ---
Target rows vs Predict columns (0: Normal, 1: Crackle, 2: Wheeze, 3: Both)
[[2572  735  259   76]
 [1127  590  103   44]
 [ 475  138  182   91]
 [ 227  128  115   36]]

--- 4-CLASS EVALUATION METRICS ---
True Negatives (TN) : 2572 (Healthy recognized as Healthy)
True Positives (TP) : 808 (Anomalies correctly classified into their EXACT class)
False Positives (FP): 1070 (Healthy wrongly flagged as Anomaly)
False Negatives (FN): 1829 (Anomalies missed or misclassified)

--- FINAL COMPUTED METRICS vs PAPER ---
Computed Sensitivity (Sen): 30.64%  | Target vs AlexNet: 45.12%
Computed Specificity (Spe): 70.62%  | Target vs AlexNet: 59.74%
Computed Score (Sco)      : 50.63%  | Target vs AlexNet: 52.43%


In [7]:
# ---------------------------------------------------------
# 5. Wheeze & Crackle Specific Evaluation (Table 6 Format)
# ---------------------------------------------------------
# The paper evaluates Wheezes and Crackles independently.
# - Wheeze present in: Class 2 (Wheeze) and Class 3 (Both)
# - Crackle present in: Class 1 (Crackle) and Class 3 (Both)
# ---------------------------------------------------------

import numpy as np

targets = np.array(global_targets)
preds = np.array(global_preds)

def calc_binary_metrics(y_true, y_pred):
    """Calculate Sen, Spe, Score, Precision for binary classification"""
    TP = np.sum((y_true == 1) & (y_pred == 1))
    TN = np.sum((y_true == 0) & (y_pred == 0))
    FP = np.sum((y_true == 0) & (y_pred == 1))
    FN = np.sum((y_true == 1) & (y_pred == 0))

    sen = TP / (TP + FN + 1e-8)
    spe = TN / (TN + FP + 1e-8)
    sco = (sen + spe) / 2.0
    pre = TP / (TP + FP + 1e-8)

    return sen * 100, spe * 100, sco * 100, pre * 100

# --- Wheeze Binary Mapping ---
w_true = ((targets == 2) | (targets == 3)).astype(int)
w_pred = ((preds == 2) | (preds == 3)).astype(int)
w_sen, w_spe, w_sco, w_pre = calc_binary_metrics(w_true, w_pred)

# --- Crackle Binary Mapping ---
c_true = ((targets == 1) | (targets == 3)).astype(int)
c_pred = ((preds == 1) | (preds == 3)).astype(int)
c_sen, c_spe, c_sco, c_pre = calc_binary_metrics(c_true, c_pred)

print("\n" + "=" * 90)
print("ALEXNET (COCHLEOGRAM) - SPECIFIC METRICS TABLE (Paper Table 6 Format)")
print("=" * 90)
print(f"{'Metric':<20} | {'Wheezes':<35} | {'Crackles':<35}")
print("-" * 90)
print(f"{'Sensitivity (Sen)':<20} | {w_sen:>6.2f}% (Paper: 66.3%) | {c_sen:>6.2f}% (Paper: 55.1%)")
print(f"{'Specificity (Spe)':<20} | {w_spe:>6.2f}% (Paper: 72.1%) | {c_spe:>6.2f}% (Paper: 62.7%)")
print(f"{'Score (Sco)':<20} | {w_sco:>6.2f}% (Paper: 69.2%) | {c_sco:>6.2f}% (Paper: 58.9%)")
print(f"{'Precision (Pre)':<20} | {w_pre:>6.2f}% (Paper: 44.4%) | {c_pre:>6.2f}% (Paper: 44.4%)")
print("=" * 90)


ALEXNET (COCHLEOGRAM) - SPECIFIC METRICS TABLE (Paper Table 6 Format)
Metric               | Wheezes                             | Crackles                           
------------------------------------------------------------------------------------------
Sensitivity (Sen)    |  30.46% (Paper: 66.3%) |  33.67% (Paper: 55.1%)
Specificity (Spe)    |  91.25% (Paper: 72.1%) |  77.03% (Paper: 62.7%)
Score (Sco)          |  60.85% (Paper: 69.2%) |  55.35% (Paper: 58.9%)
Precision (Pre)      |  46.80% (Paper: 44.4%) |  43.42% (Paper: 44.4%)
